In [1]:
## 0 – Imports & basic configuration
import os, json, math, re
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
from discovery_utils.utils.llm import batch_check
from discovery_heat_pump_futures import PROJECT_DIR
   


/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import OpenAI
load_dotenv()
client = OpenAI()

In [3]:
## 1 – Load the full patent & paper datasets 
OPENALEX_URL = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
PATENT_URL   = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json"

openalex_df = pd.read_csv(OPENALEX_URL, low_memory=False)
patents_df  = pd.read_json(PATENT_URL, lines=True)

def format_title_abstract(df, title="title", abstract="abstract"):
    df = df.copy()
    df["title_abstract"] = (
        "TITLE: " + df[title].fillna("").str.lower() +
        " ABSTRACT: " + df[abstract].fillna("").str.lower()
    )
    return df

openalex_df = format_title_abstract(openalex_df)
patents_df  = format_title_abstract(patents_df)


print(f"Loaded {len(openalex_df):,} papers | {len(patents_df):,} patents")

patents_df.head()

Loaded 33,599 papers | 45,957 patents


,publication_number,application_number,family_id,url,filing_date,grant_date,publication_date,priority_date,title,title_translated,abstract,abstract_translated,top_terms,country_code,inventor_harmonized,assignee_harmonized,cpc,title_abstract
0,WO-2024184245-A1,EP-2024055431-W,85462233,https://patents.google.com/patent/WO2024184245A1,20240301,0,20240912,20230303,Heat source unit of an air heat pump,False,"Heat source unit (1) of an air heat pump, havi...",False,"[fan, louvers, heat source, source unit, cente...",WO,"[{'name': 'HEINTZ NICOLAS', 'country_code': 'B...","[{'name': 'DAIKIN EUROPE NV', 'country_code': ...","[{'code': 'F24F13/082', 'inventive': False, 'f...",TITLE: heat source unit of an air heat pump AB...
1,US-11994321-B2,US-202017420370-A,71406811,https://patents.google.com/patent/US11994321B2,20200103,20240528,20240528,20190103,High performance compressors and vapor compres...,False,The present disclosure relates to a new breed ...,False,"[compressor, pressure, motor, refrigerant, pum...",US,"[{'name': 'LEE KANG P', 'country_code': 'US'},...","[{'name': 'ASPEN COMPRESSOR LLC', 'country_cod...","[{'code': 'F25B31/026', 'inventive': True, 'fi...",TITLE: high performance compressors and vapor ...
2,WO-2022023086-A1,EP-2021070024-W,71846293,https://patents.google.com/patent/WO2022023086A1,20210716,0,20220203,20200729,A heating system,False,A heating system (1) has a turbine (20) for bu...,False,"[flue gas, heat exchanger, air, wshp, water, h...",WO,"[{'name': 'FLYNN LIAM', 'country_code': 'IE'},...","[{'name': 'ENERGY SERVICES LTD', 'country_code...","[{'code': 'Y02B30/18', 'inventive': False, 'fi...",TITLE: a heating system ABSTRACT: a heating sy...
3,CN-211854451-U,CN-202020379578-U,73135275,https://patents.google.com/patent/CN211854451U,20200323,20201103,20201103,20200323,Cold and hot water supply system for poultry s...,False,The utility model provides a poultry is slaugh...,False,"[hot water, cold, water pipe, supply system, p...",CN,"[{'name': 'WU TAN'EN', 'country_code': ''}]",[{'name': 'TAIZHOU HONGYE ENVIRONMENTAL TECH C...,[],TITLE: cold and hot water supply system for po...
4,WO-2024162444-A1,JP-2024003342-W,91030893,https://patents.google.com/patent/WO2024162444A1,20240201,0,20240808,20230202,Heat pump device,False,A heat pump device according to one embodiment...,False,"[discharge, temperature, compressor, pressure,...",WO,"[{'name': 'SAGAWA KENTARO', 'country_code': 'J...","[{'name': 'FUJITSU GENERAL LTD', 'country_code...","[{'code': 'F25B1/00', 'inventive': True, 'firs...",TITLE: heat pump device ABSTRACT: a heat pump ...


In [4]:
## 2.1 – Heat-pump taxonomy & keyword universe
CATEGORIES = {
    "1.1": "Compressors", 
    "1.2": "Refrigerants", 
    "1.3": "Heat-exchangers", 
    "1.4": "Motor & Drives", 
    "1.5": "Lubrication & oil management",
    "2.1": "Elastocaloric", 
    "2.2": "Electrocaloric", 
    "2.3": "Magnetocaloric", 
    "2.4": "Ionocaloric", 
    "2.5": "Barocaloric",
    "2.6": "Thermoelectric", 
    "2.7": "Electro- & Chemisorption", 
    "2.8": "Thermoacoustic", 
    "2.9": "Sorption / Absorption", 
    "2.10": "Hybrid & cascade",
    "3.1": "Commissioning & installation",  # Changed from "Topology & configuration"
    "3.2": "Controls & optimisation", 
    "3.3": "Operational integration",
    "4.1": "Flexible cycles", 
    "4.2": "Defrost & icing mitigation", 
    "4.3": "Thermal storage",
    "5.1": "Design-for-disassembly", 
    "5.2": "Modular assemblies", 
    "5.3": "Recycled materials", 
    "5.4": "Additive manufacturing", 
    "5.5": "Predictive maintenance"
}

# ---- 2.2 KEYWORD UNIVERSE ---------------------------------------------------
KWS = {
    "1.1": ["scroll", "rotary", "vane", "twin-screw", "reciprocating", "isothermal compressor", "oil-free", "variable-speed", "economiser", "compressor-modulation", "two-stage"],
    "1.2": ["R32", "R454B", "R290", "propane", "CO₂", "carbon dioxide", "low-GWP", "natural refrigerant", "azeotrope", "zeotropic", "ionic liquid"],
    "1.3": ["micro-channel", "plate heat exchanger", "fin-tube", "enthalpy exchanger", "phase-change heat exchanger", "anti-fouling", "frost-free", "3-D printed"],
    "1.4": ["IPM motor", "SiC inverter", "PMSM", "sensor-less", "flux-weakening"],
    "1.5": ["oil-separator", "mist injection", "CRII", "low-viscosity oil"],
    "2.1": ["elastocaloric", "shape-memory", "Ni-Ti"], 
    "2.2": ["electrocaloric", "ferroelectric"], 
    "2.3": ["magnetocaloric", "gadolinium"],
    "2.4": ["ionocaloric", "ion-solvation"], 
    "2.5": ["barocaloric"], 
    "2.6": ["thermoelectric", "Peltier", "Seebeck"],
    "2.7": ["electrochemical compressor", "chemisorption"], 
    "2.8": ["thermoacoustic"],
    "2.9": ["adsorption heat pump", "absorption heat pump", "lithium bromide"], 
    "2.10": ["cascade heat pump", "hybrid heat pump"],
    "3.1": ["commissioning", "installation", "setup", "configuration", "system design", "cascade", "booster", "trans-critical", "transcritical", "bi-valent", "bivalent", "ground-source", "ground source"],
    "3.2": ["model predictive control", "MPC", "PID", "fault detection", "digital twin", "optimization", "machine learning", "AI control", "smart control"],
    "3.3": ["smart-grid", "thermal district network", "hybrid boiler"], 
    "4.1": ["ejector cycle", "regenerative cycle", "parallel-compressor"],
    "4.2": ["defrost", "icing sensor", "hot-gas bypass", "nano-coating"], 
    "4.3": ["PCM storage", "phase change material", "heat battery", "stratified tank"],
    "5.1": ["tool-less", "snap-fit", "reversible adhesive", "design-for-disassembly", "DfD"], 
    "5.2": ["modular cartridge", "remanufacture", "refurbish", "life-extension"],
    "5.3": ["recycled", "bio-polymer", "PCR plastic", "reclaimed copper", "low-GWP foam"], 
    "5.4": ["additive manufacturing", "3-D print", "near-net shape", "binder-jet"],
    "5.5": ["predictive maintenance", "condition monitoring", "service-as-a-product", "serviceability", "repare", "restore"]
}

#Old KW2CAT = {kw.lower(): cat for cat, kwlist in KWS.items() for kw in kwlist}

#Rosie's feedback Slack
# Normalize keywords:
def normalize_keyword(kw):
    """Normalize keywords to handle variants (e.g., CO₂ -> co2)"""
    return kw.lower().replace("₂", "2").replace("-", " ").replace("_", " ")

# Updated KW2CAT creation to use normalized keywords:
KW2CAT = {}
for cat, kwlist in KWS.items():
    for kw in kwlist:
        KW2CAT[normalize_keyword(kw)] = cat

# Function to check if text contains any keyword:
def contains_keyword(text):
    """Check if text contains any heat pump keyword"""
    text_normalized = normalize_keyword(text)
    return any(normalize_keyword(kw) in text_normalized for kw_list in KWS.values() for kw in kw_list)


In [5]:
#2.3 Category Explanation for accuracy
CATEGORY_EXPLANATIONS = {
    # Traditional Components 
    "1.1": "Compressors: Devices that compress refrigerant, including scroll, rotary, reciprocating types",
    "1.2": "Refrigerants: Working fluids that undergo phase changes, including natural and synthetic options",
    "1.3": "Heat-exchangers: Components for heat transfer between refrigerant and air/water",
    "1.4": "Motor & Drives: Electric motors and control systems that power the compressor",
    "1.5": "Lubrication & oil management: Systems for compressor lubrication and oil circulation",
    
    # Non-traditional Technologies 
    "2.1": "Elastocaloric: Uses stress-induced phase transitions in shape-memory alloys",
    "2.2": "Electrocaloric: Uses electric field-induced temperature changes in ferroelectric materials",
    "2.3": "Magnetocaloric: Uses magnetic field-induced temperature changes (e.g., gadolinium)",
    "2.4": "Ionocaloric: Uses ion dissolution/crystallization for cooling",
    "2.5": "Barocaloric: Uses pressure-induced phase transitions",
    "2.6": "Thermoelectric: Uses Peltier/Seebeck effects for solid-state cooling",
    "2.7": "Electro- & Chemisorption: Uses electrochemical processes or chemical absorption",
    "2.8": "Thermoacoustic: Uses acoustic waves to create temperature differences",
    "2.9": "Sorption/Absorption: Uses chemical absorption (e.g., lithium bromide-water)",
    "2.10": "Hybrid & cascade: Combines multiple technologies or stages",
    
    # System Design 
    "3.1": "Commissioning & installation: System setup, configuration, and initial startup procedures",
    "3.2": "Controls & optimisation: Control systems, algorithms, and performance optimization",
    "3.3": "Operational integration: Integration with buildings, grids, or other systems",
    
    # System Enhancements 
    "4.1": "Flexible cycles: Advanced refrigeration cycles for improved performance",
    "4.2": "Defrost & icing mitigation: Technologies to prevent or remove ice formation",
    "4.3": "Thermal storage: Heat/cold storage systems for load shifting",
    
    # Circular Economy 
    "5.1": "Design-for-disassembly: Products designed for easy dismantling and component recovery",
    "5.2": "Modular assemblies: Replaceable/upgradeable modules for extended product life",
    "5.3": "Recycled materials: Use of recycled or sustainable materials in manufacturing",
    "5.4": "Additive manufacturing: 3D printing and other advanced manufacturing techniques",
    "5.5": "Predictive maintenance: IoT and AI-based maintenance to extend equipment life"
}

In [6]:
# Karlis's feedback: Defining APPLICATION_INDICATORS (as a dictionary)
APPLICATION_INDICATORS = {
    "domestic": ["residential", "home", "household", "domestic", "small-scale", "<20kW", "single-family", "apartment"],
    "industrial": ["commercial", "industrial", "large-scale", "district", "process heat", ">100kW", "manufacturing", "warehouse"]
}

# Function to format the indicators
def format_application_indicators(indicators_dict):
    """Format application indicators for display in prompt"""
    lines = []
    for app_type, keywords in indicators_dict.items():
        lines.append(f"- {app_type.capitalize()}: {', '.join(keywords)}")
    return "\n".join(lines)

In [7]:
# Build the category strings BEFORE the f-string to avoid nested braces

# Convert CATEGORIES to a clean string format
categories_str = "\n".join([f"{key}: {value}" for key, value in CATEGORIES.items()])

# Convert CATEGORY_EXPLANATIONS to a clean string format  
explanations_str = "\n".join([f"{key}: {value}" for key, value in CATEGORY_EXPLANATIONS.items()])

# Get application indicators string
app_indicators_str = format_application_indicators(APPLICATION_INDICATORS)

# Now build the system message with the pre-formatted strings
SYSTEM_MESSAGE_1 = f"""You are a sustainable-heating technology analyst evaluating heat pump innovations.

CATEGORIES FOR CLASSIFICATION:
{categories_str}

CATEGORY EXPLANATIONS:
{explanations_str}

Application Type Indicators:
{app_indicators_str}

INSTRUCTIONS:
Analyze the patents and papers documents and provide a comprehensive filtering following the field definitions exactly.

You must classify each document into ONE of the categories listed above. Use the category explanations to guide your classification.

Return ONLY valid JSON matching the specified fields."""

# Also update FIELDS_1 to reference the categories properly
FIELDS_1 = [
    # Relevance and classification
    {"name": "is_relevant", "type": "str", 
     "description": "One-word answer: 'yes' if the text is about heat pump innovation or technology, otherwise 'no'."},
    {"name": "relevance_reason", "type": "str", 
     "description": "Short explanation (one sentence, ≤20 words) of why the text is relevant or not."},
    
    # Basic information ## ONE primary category
    {"name": "summary", "type": "str", 
     "description": "A brief summary (≤25 words) of the innovation or technology described."},
    {"name": "category", "type": "str", 
     "description": "Select ONE primary category from the CATEGORIES FOR CLASSIFICATION list above. Use the exact category name (e.g., 'Compressors', 'Refrigerants', etc.)."},
    
    # Application context
    {"name": "application_type", "type": "str", 
     "description": "Identify the application: 'domestic' (residential, <20kW), 'industrial' (commercial, >100kW), 'both', or 'unclear'."},
    {"name": "specific_applications", "type": "list[str]", 
     "description": "Specific use cases mentioned (e.g., 'space heating', 'water heating', 'process heat', 'district heating')."},
]

# Debug: Print a sample to verify the format
print("=== SAMPLE OF SYSTEM MESSAGE ===")
print(SYSTEM_MESSAGE_1[:500] + "...")
print("\n=== CATEGORIES SECTION ===")
print(categories_str[:200] + "...")

=== SAMPLE OF SYSTEM MESSAGE ===
You are a sustainable-heating technology analyst evaluating heat pump innovations.

CATEGORIES FOR CLASSIFICATION:
1.1: Compressors
1.2: Refrigerants
1.3: Heat-exchangers
1.4: Motor & Drives
1.5: Lubrication & oil management
2.1: Elastocaloric
2.2: Electrocaloric
2.3: Magnetocaloric
2.4: Ionocaloric
2.5: Barocaloric
2.6: Thermoelectric
2.7: Electro- & Chemisorption
2.8: Thermoacoustic
2.9: Sorption / Absorption
2.10: Hybrid & cascade
3.1: Commissioning & installation
3.2: Controls & optimisation...

=== CATEGORIES SECTION ===
1.1: Compressors
1.2: Refrigerants
1.3: Heat-exchangers
1.4: Motor & Drives
1.5: Lubrication & oil management
2.1: Elastocaloric
2.2: Electrocaloric
2.3: Magnetocaloric
2.4: Ionocaloric
2.5: Barocalor...


In [8]:
patents_sample = patents_df.sample(5, random_state=42)

test_data = patents_sample[["url", "title_abstract"]]

test_dict = test_data.set_index("url")["title_abstract"].to_dict()
test_dict

{'https://patents.google.com/patent/CN222157138U': 'TITLE: a valve plate structure of an automobile thermal management system with hot gas bypass function ABSTRACT: the utility model discloses a valve plate structure of an automobile thermal management system with a hot gas bypass function, including a valve plate body, a first side of the valve plate body is provided with a plurality of flow channels, a second side of the valve plate body is provided with a first inlet and a second inlet connected to a gas-liquid separator in parallel, a first side of the valve plate body is provided with a low-temperature and low-pressure refrigerant flow channel connected to the first inlet and a medium-temperature and medium-pressure refrigerant flow channel connected to the second inlet, a second side of the valve plate body is also provided with a low-temperature and low-pressure refrigerant flow channel, the lower end of the low-temperature and low-pressure refrigerant flow channel is connected 

In [ ]:
outpath = PROJECT_DIR / "outputs/Stage_1.test.jsonl"

processor = batch_check.LLMProcessor(
    model_name="gpt-4o-mini",
    temperature=0,
    output_path=str(outpath),
    system_message=SYSTEM_MESSAGE_1,
    session_name="Stage_1_test",
    output_fields=FIELDS_1,
)

task = processor.run(test_dict, batch_size=1, sleep_time=0.5)
await task

2025-05-27 14:17:12,397 - root - INFO - Using OpenAI
2025-05-27 14:17:12,465 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-05-27 14:17:12,511 - root - INFO - Processing batch 1/5
2025-05-27 14:17:15,376 - root - INFO - Processing batch 2/5
2025-05-27 14:17:17,510 - root - INFO - Processing batch 3/5
2025-05-27 14:17:20,638 - root - INFO - Processing batch 4/5
2025-05-27 14:17:23,479 - root - INFO - Processing batch 5/5


In [ ]:
#Process Full Datasets (Papers + Patents)

# Create ID→text dictionaries for full datasets
print("Creating full dataset dictionaries...")
paper_dict = openalex_df.set_index("id")["title_abstract"].to_dict()
patent_dict = patents_df.set_index("publication_number")["title_abstract"].to_dict()

print(f"Papers to process: {len(paper_dict):,}")
print(f"Patents to process: {len(patent_dict):,}")
print(f"Total documents: {len(paper_dict) + len(patent_dict):,}")

# Combine both datasets with prefixed keys to avoid ID conflicts
combined_dict = {}

# Add papers with "paper_" prefix
for paper_id, content in paper_dict.items():
    combined_dict[f"paper_{paper_id}"] = content

# Add patents with "patent_" prefix  
for patent_id, content in patent_dict.items():
    combined_dict[f"patent_{patent_id}"] = content

print(f"Combined dataset size: {len(combined_dict):,}")

# Set up output path for full processing
outpath_full = PROJECT_DIR / "outputs/Stage_1_Full_Analysis.jsonl"

# Create processor for full dataset
processor_full = batch_check.LLMProcessor(
    model_name="gpt-4o-mini",
    temperature=0,
    output_path=str(outpath_full),
    system_message=SYSTEM_MESSAGE_1,
    session_name="Stage_1_Full",
    output_fields=FIELDS_1,
)

# Process with appropriate batch size and timing for large dataset
print(f"Starting full dataset processing...")
print(f"Output will be saved to: {outpath_full}")
print("This may take a while depending on dataset size...")

# Adjust batch_size and sleep_time for efficient processing
# Larger batch_size = faster but more API load
# Longer sleep_time = slower but gentler on API limits
task_full = processor_full.run(
    combined_dict, 
    batch_size=10,      # Process 10 documents per batch
    sleep_time=1.0      # 1 second between batches
)

# Run the full processing
await task_full

print("✅ Full dataset processing completed!")
print(f"Results saved to: {outpath_full}")

# Optional: Quick analysis of results
try:
    results_df = pd.read_json(outpath_full, lines=True)
    print(f"\n=== PROCESSING RESULTS SUMMARY ===")
    print(f"Total processed: {len(results_df):,}")
    print(f"Relevant documents: {len(results_df[results_df['is_relevant'] == 'yes']):,}")
    print(f"Success rate: {len(results_df[results_df['is_relevant'] == 'yes']) / len(results_df) * 100:.1f}%")
    
    # Category distribution
    if 'category' in results_df.columns:
        print(f"\n=== TOP CATEGORIES ===")
        category_counts = results_df[results_df['is_relevant'] == 'yes']['category'].value_counts().head(10)
        for category, count in category_counts.items():
            print(f"{category}: {count}")
            
except Exception as e:
    print(f"Could not load results for summary: {e}")

Creating full dataset dictionaries...
Papers to process: 33,599
Patents to process: 45,957
Total documents: 79,556
Combined dataset size: 79,556
2025-05-27 14:23:01,995 - root - INFO - Using OpenAI
2025-05-27 14:23:02,043 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
Starting full dataset processing...
Output will be saved to: /home/pascualdiego/projects/DiscoveryHP/outputs/Stage_1_Full_Analysis.jsonl
This may take a while depending on dataset size...
2025-05-27 14:23:02,112 - root - INFO - Processing batch 1/7956
2025-05-27 14:23:06,592 - root - INFO - Processing batch 2/7956
2025-05-27 14:23:10,694 - root - INFO - Processing batch 3/7956
2025-05-27 14:23:16,320 - root - INFO - Processing batch 4/7956
2025-05-27 14:23:27,753 - root - INFO - Processing batch 5/7956
2025-05-27 14:23:31,096 - root - INFO - P

CancelledError: 